# Step 3b: Fixing the subsampling strategy

our real results surfaced something important:

```
Hidden rows with never-before-seen user:  10,169  (~5.1% of hidden)
Hidden rows with never-before-seen movie:  1,560  (~0.8% of hidden)
```

That's a lot of cold-start cases; more than you'd expect from a
healthy 80/20 split. This is becase **subsampling by random ROW first
thins out how many ratings each user has left.**

The real dataset averages ~61.5 ratings per user. But once you randomly
keep only 500K of the 10M total rows, the *average* user now has only
about 3 of their original 61 ratings surviving in our subsample. With
just 1-3 ratings left, there's a real chance the 80/20 split sends ALL
of a user's remaining ratings into the hidden pile — making them look
like a brand new user, when in the full dataset they're not cold-start
at all.

**The fix:** subsample by USER, not by row. Pick a random subset of
users, but keep ALL of their ratings. This preserves each included
user's full rating history, so cold-start cases you see afterward are
real, not an artifact of how we sampled.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

DATA_DIR = "data"
N_USERS_TO_KEEP = 15_000   # tune this to land near any target row count

train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
print(f"Full data: {train_full.shape}")

Full data: (10000038, 4)


## Subsample by user, not by row

In [ ]:
# Select a random subset of users for the subsample.
# We use a random number generator with a fixed seed for reproducibility, and we select a random subset of users to keep in the training data.
# The number of users to keep is determined by the N_USERS_TO_KEEP constant, but we ensure that we don't try to keep more users than exist in the dataset.
# train_full["userId"].nunique() gives the number of unique users in the full training data, and we take the minimum of that and N_USERS_TO_KEEP to determine how many users to keep in the subsample.
rng = np.random.default_rng(42)
all_users = train_full["userId"].unique()
n_to_keep = min(N_USERS_TO_KEEP, len(all_users))  # don't ask for more users than exist
chosen_users = rng.choice(all_users, size=n_to_keep, replace=False)

train = train_full[train_full["userId"].isin(chosen_users)].reset_index(drop=True)

print(f"Subsample: {train.shape}")
print(f"Users kept: {train['userId'].nunique():,} (each with their FULL rating history)")
print(f"Avg ratings per user in this subsample: {len(train) / train['userId'].nunique():.1f}")

Subsample: (930202, 4)
Users kept: 15,000 (each with their FULL rating history)
Avg ratings per user in this subsample: 62.0


## Re-runing the same 80/20 split and checking the cold-start rate

This should now be much lower than before, since each included user
keeps their entire rating history — a random split is far less likely
to push ALL of someone's ratings into the hidden pile.

In [3]:
visible, hidden = train_test_split(train, test_size=0.2, random_state=42)

unseen_users  = (~hidden["userId"].isin(visible["userId"])).sum()
unseen_movies = (~hidden["movieId"].isin(visible["movieId"])).sum()

print(f"Visible: {len(visible):,}   Hidden: {len(hidden):,}")
print(f"Hidden rows with never-before-seen user:  {unseen_users} ({unseen_users/len(hidden):.2%})")
print(f"Hidden rows with never-before-seen movie: {unseen_movies} ({unseen_movies/len(hidden):.2%})")

Visible: 744,161   Hidden: 186,041
Hidden rows with never-before-seen user:  0 (0.00%)
Hidden rows with never-before-seen movie: 1670 (0.90%)


## Re-run the bias model on the cleaner subsample

Same formula as before: `global_mean + user_bias + movie_bias`,
computed from visible only, evaluated on hidden.

In [ ]:
# Compute the global mean rating, user bias, and movie bias from the visible data.
# The global mean is the average rating across all ratings in the visible set. 
# The user bias is calculated as the average rating for each user minus the global mean, and the movie bias is calculated as the average rating for each movie minus the global mean.
# We then create a copy of the hidden set and map the user bias and movie bias to the corresponding userId and movieId in the hidden set.
global_mean = visible["rating"].mean()
user_bias  = visible.groupby("userId")["rating"].mean() - global_mean
movie_bias = visible.groupby("movieId")["rating"].mean() - global_mean

hidden = hidden.copy()
hidden["user_bias"]  = hidden["userId"].map(user_bias).fillna(0)
hidden["movie_bias"] = hidden["movieId"].map(movie_bias).fillna(0)
hidden["predicted"]  = (global_mean + hidden["user_bias"] + hidden["movie_bias"]).clip(0.5, 5.0)

rmse_bias = np.sqrt(mean_squared_error(hidden["rating"], hidden["predicted"]))
rmse_mean_only = np.sqrt(mean_squared_error(hidden["rating"], np.full(len(hidden), global_mean)))

print(f"RMSE — global mean only   : {rmse_mean_only:.4f}")
print(f"RMSE — user + movie bias  : {rmse_bias:.4f}")
print(f"Improvement: {rmse_mean_only - rmse_bias:.4f}")

RMSE — global mean only   : 1.0566
RMSE — user + movie bias  : 0.8998
Improvement: 0.1568


## What to expect

With cold-start cases brought down to a realistic level, the
improvement from adding user + movie bias should now look much more
like the ~0.3 RMSE drop we saw in earlier testing, rather than the
thin ~0.05 improvement the flawed subsampling produced.

This is also a good general lesson for the rest of the project:
**how you sample matters as much as what model you build.** We'll
keep this user-based subsampling approach for SVD++ and beyond.

## Next step

True collaborative filtering with SVD++ — same hide-and-predict
validation pattern, now on a properly-sampled subset.